In [52]:
from jazzmus.dataset.eval_functions import compute_poliphony_metrics
from jazzmus.dataset.tokenizer import untokenize

In [53]:
def extract_spines(kern_text):
    """
    Extract individual spines from **kern format.

    Returns dict with spine name -> content mapping.
    E.g., {'**kern': '...melody...', '**mxhm': '...chords...'}
    """
    lines = kern_text.strip().split('\n')
    spines = {}
    spine_indices = {}

    # Find spine headers (lines starting with **)
    for i, line in enumerate(lines):
        if line.startswith('**'):
            parts = line.split('\t')
            for j, part in enumerate(parts):
                if part.startswith('**'):
                    spine_name = part
                    if spine_name not in spines:
                        spines[spine_name] = []
                        spine_indices[spine_name] = j

    # Extract content for each spine
    for line in lines:
        if line.startswith('*') or line.startswith('=') or line.startswith('!'):
            # Metadata/formatting line - include for all spines
            parts = line.split('\t')
            for spine_name, idx in spine_indices.items():
                if idx < len(parts):
                    spines[spine_name].append(parts[idx])
        else:
            # Data line
            parts = line.split('\t')
            for spine_name, idx in spine_indices.items():
                if idx < len(parts):
                    spines[spine_name].append(parts[idx])

    # Join lines back together
    result = {}
    for spine_name, content_list in spines.items():
        result[spine_name] = '\n'.join(content_list)

    return result

In [54]:
def calculate_spine_metrics(prediction, ground_truth):
    """
    Calculate CER/SER/LER for individual spines and overall.

    Returns dict with metrics for each spine and overall.
    """
    
    # Extract spines
    pred_spines = extract_spines(prediction)
    gt_spines = extract_spines(ground_truth)
    print("pred spines:", pred_spines)
    print("gt spines:", gt_spines)
    # Get all spine names
    all_spines = set(pred_spines.keys()) | set(gt_spines.keys())

    results = {}
    # Calculate metrics for each spine
    for spine_name in sorted(all_spines):
        pred_spine = pred_spines.get(spine_name, "")
        gt_spine = gt_spines.get(spine_name, "")

        if not gt_spine:
            print("no gt spine")# Skip if no ground truth for this spine
            continue

        try:
            cer, ser, ler = compute_poliphony_metrics([pred_spine], [gt_spine])
            results[spine_name] = {
                "cer": cer,
                "ser": ser,
                "ler": ler,
            }
        except Exception as e:
            results[spine_name] = {
                "cer": 100.0,
                "ser": 100.0,
                "ler": 100.0,
                "error": str(e),
            }

In [55]:
GROUND_TRUTH_PATH = "data/jazzmus_systems/gt/img_10_1.txt"

with open(GROUND_TRUTH_PATH, "r") as f:
    ground_truth = f.read()
    
extracted_spines = extract_spines(ground_truth)



In [56]:
kern = extracted_spines['**kern']
kern = untokenize(kern)
print(kern)

**kern
*clefG2
*k[f#]
*G:
!!linebreak:original
=
8a
4.b
8r
16r
16dKL
8.e
16gJk
=
8a
4.b
4g
8gL
[8gJ
=
2.g]
4r
=
4b
4cc
4cc
4b
=
4a
[4b
2b]
!!linebreak:original
*-


In [57]:
chord = extracted_spines['**mxhm']
chord = untokenize(chord)   
print(chord)

**mxhm
*
*
*
=
A:min
.
G:maj/B
.
.
.
.
=
C:maj6
G:maj/B
Bb:dim7
A:min7
.
=
G:maj
.
=
B:7
.
E:7
.
=
A:7
.
D:7
*-


In [58]:
calculate_spine_metrics(ground_truth, ground_truth)

pred spines: {'**kern': '**kern\n*clefG2\n*k[f#]\n*G:\n!!linebreak:original\n=\n8a\n4.b\n8r\n16r\n16dKL\n8.e\n16gJk\n=\n8a\n4.b\n4g\n8gL\n[8gJ\n=\n2.g]\n4r\n=\n4b\n4cc\n4cc\n4b\n=\n4a\n[4b\n2b]\n!!linebreak:original\n*-', '**mxhm': '**mxhm\n*\n*\n*\n=\nA:min\n.\nG:maj/B\n.\n.\n.\n.\n=\nC:maj6\nG:maj/B\nBb:dim7\nA:min7\n.\n=\nG:maj\n.\n=\nB:7\n.\nE:7\n.\n=\nA:7\n.\nD:7\n*-'}
gt spines: {'**kern': '**kern\n*clefG2\n*k[f#]\n*G:\n!!linebreak:original\n=\n8a\n4.b\n8r\n16r\n16dKL\n8.e\n16gJk\n=\n8a\n4.b\n4g\n8gL\n[8gJ\n=\n2.g]\n4r\n=\n4b\n4cc\n4cc\n4b\n=\n4a\n[4b\n2b]\n!!linebreak:original\n*-', '**mxhm': '**mxhm\n*\n*\n*\n=\nA:min\n.\nG:maj/B\n.\n.\n.\n.\n=\nC:maj6\nG:maj/B\nBb:dim7\nA:min7\n.\n=\nG:maj\n.\n=\nB:7\n.\nE:7\n.\n=\nA:7\n.\nD:7\n*-'}


In [59]:
def middle_level_split(line, piece_started):
    # handle non note-chord lines
    if not piece_started or "=" in line:
        elements = line.split("\t")
        tokens = []
        for element in elements:
            tokens.append(element)
            tokens.append("<t>")
        if tokens[-1] == "<t>":
            tokens.pop()
        tokens.append("<n>")
    else:
        # last token from line.split("\t") is the chord, the rest are notes
        tokens = line.split("\t")
        if len(tokens) == 1:
            # single spline, only notes
            tokens = note_split(tokens[0])
            tokens.append("<n>")
        else:
            notes = tokens[:-1]
            chord = tokens[-1]
            tokens = []
            for note in notes:
                tokens.extend(note_split(note))
                tokens.append("<t>")
            tokens.extend(chord_split(chord))
            tokens.append("<n>")
    return tokens

In [60]:
# def process_text(lines, tokenizer_type: str = "word"):
#     """Reads and processes the input file with text preprocessing and optional character-level or middle level tokenization."""

#     reserved_lines = {"!!linebreak", "!!pagebreak", "*I", "*F:", "!LO"}
#     piece_started = False
#     tokens = []

#     for line in lines:
#         # remove the X character as it does not have graphical impact
#         # TODO remove this and modify the dataset
#         line = line.replace("X", "")
#         if line[0].isdigit() or "=" in line:
#             piece_started = True

#         # Skip reserved lines
#         if any(reserved in line for reserved in reserved_lines):
#             continue

#         if tokenizer_type == "medium":
#             tokens.extend(middle_level_split(line.replace("\n", ""), piece_started))
#         else:
#             line_elements = line.replace("\n", "").split("\t")

#             if tokenizer_type == "character":
#                 for element in line_elements:
#                     for char in element:
#                         tokens.append(char)
#                     tokens.append("<t>")
#             elif tokenizer_type == "word":
#                 for element in line_elements:
#                     tokens.append(element)
#                     tokens.append("<t>")
#             else:
#                 raise ValueError(f"Unknown tokenizer type: {tokenizer_type}")

#             # Remove the last tab token
#             if tokens[-1] == "<t>":
#                 tokens.pop()
#             tokens.append("<n>")

#     return tokens

In [61]:
from jazzmus.dataset.tokenizer import untokenize, process_text
ground_truth_tokens = process_text(ground_truth, tokenizer_type="medium")
print(len(ground_truth_tokens))
ground_truth_tokens

610


['*',
 '<n>',
 '*',
 '<n>',
 'k',
 '<n>',
 'e',
 '<n>',
 'r',
 '<n>',
 'n',
 '<n>',
 '',
 '<t>',
 '',
 '<n>',
 '*',
 '<n>',
 '*',
 '<n>',
 'm',
 '<n>',
 'x',
 '<n>',
 'h',
 '<n>',
 'm',
 '<n>',
 '',
 '<n>',
 '*',
 '<n>',
 'c',
 '<n>',
 'l',
 '<n>',
 'e',
 '<n>',
 'f',
 '<n>',
 'G',
 '<n>',
 '2',
 '<n>',
 '<t>',
 '<chord-pitch>',
 ':',
 'none',
 '<n>',
 '*',
 '<n>',
 '<n>',
 '*',
 '<n>',
 'k',
 '<n>',
 '[',
 '<n>',
 'f',
 '<n>',
 '#',
 '<n>',
 ']',
 '<n>',
 '<t>',
 '<chord-pitch>',
 ':',
 'none',
 '<n>',
 '*',
 '<n>',
 '<n>',
 '*',
 '<n>',
 'G',
 '<n>',
 ':',
 '<n>',
 '<t>',
 '<chord-pitch>',
 ':',
 'none',
 '<n>',
 '*',
 '<n>',
 '<n>',
 '!',
 '<n>',
 '!',
 '<n>',
 'l',
 '<n>',
 'i',
 '<n>',
 'n',
 '<n>',
 'e',
 '<n>',
 'b',
 '<n>',
 'r',
 '<n>',
 'e',
 '<n>',
 'a',
 '<n>',
 'k',
 '<n>',
 ':',
 '<n>',
 'o',
 '<n>',
 'r',
 '<n>',
 'i',
 '<n>',
 'g',
 '<n>',
 'i',
 '<n>',
 'n',
 '<n>',
 'a',
 '<n>',
 'l',
 '<n>',
 '<n>',
 '=',
 '<n>',
 '<t>',
 '<chord-pitch>',
 ':',
 'none',
 '<n>',
 '=',

In [62]:
untokenize(ground_truth_tokens)
ground_truth_tokens

['*',
 '<n>',
 '*',
 '<n>',
 'k',
 '<n>',
 'e',
 '<n>',
 'r',
 '<n>',
 'n',
 '<n>',
 '',
 '<t>',
 '',
 '<n>',
 '*',
 '<n>',
 '*',
 '<n>',
 'm',
 '<n>',
 'x',
 '<n>',
 'h',
 '<n>',
 'm',
 '<n>',
 '',
 '<n>',
 '*',
 '<n>',
 'c',
 '<n>',
 'l',
 '<n>',
 'e',
 '<n>',
 'f',
 '<n>',
 'G',
 '<n>',
 '2',
 '<n>',
 '<t>',
 '<chord-pitch>',
 ':',
 'none',
 '<n>',
 '*',
 '<n>',
 '<n>',
 '*',
 '<n>',
 'k',
 '<n>',
 '[',
 '<n>',
 'f',
 '<n>',
 '#',
 '<n>',
 ']',
 '<n>',
 '<t>',
 '<chord-pitch>',
 ':',
 'none',
 '<n>',
 '*',
 '<n>',
 '<n>',
 '*',
 '<n>',
 'G',
 '<n>',
 ':',
 '<n>',
 '<t>',
 '<chord-pitch>',
 ':',
 'none',
 '<n>',
 '*',
 '<n>',
 '<n>',
 '!',
 '<n>',
 '!',
 '<n>',
 'l',
 '<n>',
 'i',
 '<n>',
 'n',
 '<n>',
 'e',
 '<n>',
 'b',
 '<n>',
 'r',
 '<n>',
 'e',
 '<n>',
 'a',
 '<n>',
 'k',
 '<n>',
 ':',
 '<n>',
 'o',
 '<n>',
 'r',
 '<n>',
 'i',
 '<n>',
 'g',
 '<n>',
 'i',
 '<n>',
 'n',
 '<n>',
 'a',
 '<n>',
 'l',
 '<n>',
 '<n>',
 '=',
 '<n>',
 '<t>',
 '<chord-pitch>',
 ':',
 'none',
 '<n>',
 '=',

In [64]:
# Define reserved lines to skip (same as in tokenizer.py)

with open(GROUND_TRUTH_PATH, 'r', encoding='utf-8') as f:
    lines = f.readlines()

reserved_lines = {"!!linebreak", "!!pagebreak", "*I", "*F:", "!LO"}

# Filter out:
    # 1. Reserved lines
    # 2. Spine header lines (starting with **)
filtered_lines = []
for line in lines:
    # Skip reserved lines
    if any(reserved in line for reserved in reserved_lines):
        continue
    
    filtered_lines.append(line)

    # Convert filtered lines back to text
    gt_readable = "".join(filtered_lines)

gt_readable = untokenize(gt_readable)
gt_readable

'**kern\t**mxhm\n*clefG2\t*\n*k[f#]\t*\n*G:\t*\n=\t=\n8a\tA:min\n4.b\t.\n8r\tG:maj/B\n16r\t.\n16dKL\t.\n8.e\t.\n16gJk\t.\n=\t=\n8a\tC:maj6\n4.b\tG:maj/B\n4g\tBb:dim7\n8gL\tA:min7\n[8gJ\t.\n=\t=\n2.g]\tG:maj\n4r\t.\n=\t=\n4b\tB:7\n4cc\t.\n4cc\tE:7\n4b\t.\n=\t=\n4a\tA:7\n[4b\t.\n2b]\tD:7\n*-\t*-\n'

In [ ]:
tokens = process_text(lines, tokenizer_type="mediu,")
